# Huấn luyện mô hình dự báo giá trên Google Colab (GPU)

Notebook này chỉ là **trình chạy mỏng** (CLAUDE.md mục 10: code thật nằm trong module `.py`). Nó giải nén gói code + dataset đã đóng bằng
`ml/scripts/package_for_colab.py` rồi gọi `ml/scripts/train_models.py`. Không cần MySQL/backend.

**Chuẩn bị trên máy chính (một lần cho mỗi dataset):**
```
python ml/scripts/package_for_colab.py --dataset-dir outputs/datasets/<dataset_version>
```
→ tải `ml_train_pkg_*.zip`, `dataset_<dataset_version>.zip` và `COLAB_MANIFEST.json` (thư mục `outputs/colab/`) lên Google Drive, cùng một thư mục (`DRIVE_DIR` bên dưới).

Chạy từng ô theo thứ tự. Runtime → Change runtime type → GPU (T4/L4/A100). Chỉ XGBoost dùng GPU; Random Forest và Ridge luôn chạy CPU.

In [ ]:
# 1) Tham số — sửa 3 dòng này
DRIVE_DIR = "/content/drive/MyDrive/hotel_price_intelligence"   # nơi bạn đã tải 2 file zip + COLAB_MANIFEST.json lên
DATASET_VERSION = "ds_XXXXXXXX_xxx"                               # tên thư mục dataset (khớp tên file dataset_<...>.zip)
HORIZONS = "1,3,7,14"                                             # horizon (ngày) cần huấn luyện
MODELS = "ridge,rf,xgb"                                           # baseline luôn chạy; chọn thêm ridge/rf/xgb

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi -L

In [ ]:
# 2) Kiểm toàn vẹn (SHA-256 so với COLAB_MANIFEST.json) rồi giải nén vào /content
import glob, hashlib, json, os, zipfile

manifest = json.load(open(f"{DRIVE_DIR}/COLAB_MANIFEST.json", encoding="utf-8"))
for name, info in manifest["archives"].items():
    path = f"{DRIVE_DIR}/{name}"
    digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
    assert digest == info["sha256"], f"SHA-256 lệch: {name}"
    zipfile.ZipFile(path).extractall("/content" if name.startswith("ml_train_pkg") else "/content/datasets")
    print("OK", name, f"{info['bytes'] / 1e6:.1f} MB")
assert os.path.exists(f"/content/datasets/{DATASET_VERSION}/samples.parquet"), "không thấy samples.parquet — kiểm DATASET_VERSION"
print(glob.glob("/content/ml/*"))

In [ ]:
# 3) Cài thư viện còn thiếu (Colab đã có pandas/numpy/scikit-learn/pyarrow)
!pip -q install -r /content/ml/requirements-train.txt
import sklearn, xgboost
print("sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)

In [ ]:
# 4) Huấn luyện + đánh giá (test chỉ tính một lần cho mô hình đã chọn theo validation)
OUT_ROOT = f"{DRIVE_DIR}/models"
!python /content/ml/scripts/train_models.py --dataset-dir /content/datasets/{DATASET_VERSION} --horizons {HORIZONS} --models {MODELS} --device cuda --output-root {OUT_ROOT}

In [ ]:
# 5) Tóm tắt kết quả của lần chạy mới nhất (đọc h{k}_report.json)
import json, pathlib
import pandas as pd

run_dir = sorted(pathlib.Path(OUT_ROOT, DATASET_VERSION).glob("run_*"))[-1]
rows = []
for path in sorted(run_dir.glob("h*_report.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    row = {"horizon": r["horizon"], "status": r["status"], "evaluation": r["evaluation_status"], "selected": r.get("selected_model")}
    if r.get("selected_model"):
        t = r["results"][r["selected_model"]]["test"]
        p = r["results"]["persistence"]["test"]
        row.update(acc20_test=round(t["accuracy_at_tol"], 3), mae_test=round(t["mae"]), persistence_acc20=round(p["accuracy_at_tol"], 3),
                   meets_80pct=r["meets_project_target_accuracy_at_20pct"], n_test=t["n"])
    else:
        row["reason"] = r.get("reason")
    rows.append(row)
print(run_dir)
pd.DataFrame(rows)